# 01 — Ингест: landing + Kafka → bronze
Первый слой медальона. Читаем источники и складываем **сырьё как есть** в bronze (Parquet в MinIO):- батч (клиенты, счета) из `landing/*.csv`- поток (транзакции) из Kafka topic `transactions`

## 1. Spark-сессия
Пакеты (S3A, Kafka, Delta) подхватятся из `spark-defaults.conf`.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

spark = SparkSession.builder.appName("ingestion").getOrCreate()
print("Spark:", spark.version)

Spark: 3.5.3


## 2. Батч: CSV из landing → bronze
Читаем файлы клиентов и счетов, пишем в bronze как Parquet.

In [2]:
# Клиенты
clients = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("file:///home/jovyan/work/landing/clients.csv"))

clients.write.mode("overwrite").parquet("s3a://bronze/clients")
print("clients  ->", clients.count(), "строк")

# Счета
accounts = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("file:///home/jovyan/work/landing/accounts.csv"))

accounts.write.mode("overwrite").parquet("s3a://bronze/accounts")
print("accounts ->", accounts.count(), "строк")

clients  -> 10300 строк
accounts -> 15419 строк


## 3. Поток: Kafka → bronze (Structured Streaming)

Читаем topic `transactions`. Значение в Kafka — JSON в байтах, поэтому:
1. приводим `value` к строке, 2. парсим по схеме `from_json`, 3. пишем в bronze.

`trigger(availableNow=True)` — обработать всё накопленное и **остановиться**.

> **Важно про повторные запуски.** Kafka хранит события (топик append-only),
> а `checkpointLocation` помнит, что уже прочитано. Если вы запускали генератор
> (`00`) повторно, он **дослал** новые события — и bronze будет расти. Это нормально
> для сырого слоя (bronze хранит всё как пришло, включая повторы). Лишние дубли
> уберёт silver на следующем шаге (дедуп по `tx_id`). Чтобы начать с чистого листа —
> удалите в MinIO папки `bronze/transactions` и `bronze/_checkpoints` перед прогоном.

In [3]:
# Схема события транзакции
tx_schema = StructType([
    StructField("tx_id",      LongType()),
    StructField("account_id", LongType()),
    StructField("amount",     DoubleType()),
    StructField("tx_type",    StringType()),
    StructField("merchant",   StringType()),
    StructField("status",     StringType()),
    StructField("ts",         StringType()),
])

# Чтение потока из Kafka
raw = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:19092")
    .option("subscribe", "transactions")
    .option("startingOffsets", "earliest")
    .load())

# value (bytes) -> string -> распарсить JSON по схеме
parsed = (raw
    .select(F.from_json(F.col("value").cast("string"), tx_schema).alias("d"))
    .select("d.*"))

# Запись в bronze; availableNow = прочитать всё накопленное и завершиться
query = (parsed.writeStream
    .format("parquet")
    .option("path", "s3a://bronze/transactions")
    .option("checkpointLocation", "s3a://bronze/_checkpoints/transactions")
    .trigger(availableNow=True)
    .outputMode("append")
    .start())

query.awaitTermination()
print("Поток обработан, транзакции записаны в bronze")

Поток обработан, транзакции записаны в bronze


## 4. Проверка bronze
Читаем обратно из MinIO и считаем строки.

In [4]:
for name in ["clients", "accounts", "transactions"]:
    df = spark.read.parquet(f"s3a://bronze/{name}")
    print(f"{name:13s} -> {df.count():>7} строк")

print()
spark.read.parquet("s3a://bronze/transactions").show(5, truncate=False)

clients       ->   10300 строк
accounts      ->   15419 строк
transactions  ->  205825 строк

+-----+----------+--------+--------+---------+-------+--------------------------+
|tx_id|account_id|amount  |tx_type |merchant |status |ts                        |
+-----+----------+--------+--------+---------+-------+--------------------------+
|1    |4295      |4170.11 |purchase|Яндекс   |pending|2026-05-24T14:40:46.200850|
|2    |5254      |11882.95|transfer|DNS      |failed |2026-05-24T00:29:43.415408|
|3    |12673     |200.02  |refund  |DNS      |pending|2026-05-24T01:07:48.415496|
|4    |2735      |38949.58|refund  |Аптека   |pending|2026-05-23T23:23:04.415528|
|NULL |14741     |41260.44|refund  |Пятёрочка|failed |2026-05-24T16:06:17.415552|
+-----+----------+--------+--------+---------+-------+--------------------------+
only showing top 5 rows

